In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Load v3 dataset
df = pd.read_csv("data/Training_Data_v4_noisy.csv")
df = df.dropna(subset=['graph_id'])
df['graph_id'] = df['graph_id'].astype(int)

print(f"Total rows: {len(df)}")
print(f"Unique graphs: {df['graph_id'].nunique()}")
print(f"Topologies: {df['topology'].value_counts().to_dict()}")
print(f"Zones: {df['zone'].value_counts().to_dict()}")
print(f"KEV rate: {df['in_kev'].mean()*100:.2f}%")
print(f"\nSample:")
df[['graph_id', 'topology', 'zone', 'asset_criticality', 'epss_n', 'cvss_n', 'av_n', 'in_kev']].head()

Total rows: 9868
Unique graphs: 70
Topologies: {'enterprise': 5382, 'cloud': 4486}
Zones: {'internal': 4251, 'critical': 4032, 'edge': 1075, 'dmz': 510}
KEV rate: 9.93%

Sample:


,graph_id,topology,zone,asset_criticality,epss_n,cvss_n,av_n,in_kev
0,0,cloud,edge,0.1,0.051528,0.675651,0.586088,0
1,0,cloud,edge,0.1,0.008238,0.244216,0.437736,0
2,0,cloud,edge,0.1,0.090687,0.643542,0.172202,0
3,0,cloud,edge,0.1,0.374496,0.848232,0.327583,1
4,0,cloud,edge,0.1,0.000000,0.315835,0.133145,0


In [3]:

print("STAGE 1: TRAINING XGBOOST (WITH EPSS BASELINE)")


# ═════════════════════════════════════════════════
# BASELINE: EPSS ONLY
# ═════════════════════════════════════════════════

print("\n1. BASELINE: EPSS ONLY")
print("-" * 70)

X_epss = df[['epss_n']]
y = df['in_kev']
groups = df['graph_id']

gkf = GroupKFold(n_splits=5)
clf_epss = XGBClassifier(n_estimators=100, max_depth=4, 
                         eval_metric='logloss', random_state=42, verbose=0)

oof_epss = np.zeros(len(df))
fold_aucs_epss = []

print("Cross-Validation (5-fold, grouped by graph):")
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X_epss, y, groups), 1):
    clf_epss.fit(X_epss.iloc[train_idx], y.iloc[train_idx])
    oof_epss[test_idx] = clf_epss.predict_proba(X_epss.iloc[test_idx])[:, 1]
    fold_auc = roc_auc_score(y.iloc[test_idx], oof_epss[test_idx])
    fold_aucs_epss.append(fold_auc)
    print(f"  Fold {fold_num}: AUC = {fold_auc:.4f}")

print(f"\n✓ EPSS Only Results:")
print(f"  Mean AUC: {np.mean(fold_aucs_epss):.4f}")
print(f"  Std AUC:  {np.std(fold_aucs_epss):.4f}")

df['score_epss_only'] = oof_epss

# ═════════════════════════════════════════════════
# FULL MODEL: EPSS + CVSS + AV
# ═════════════════════════════════════════════════

print("\n\n2. FULL MODEL: EPSS + CVSS + AV")
print("-" * 70)

X = df[['epss_n', 'cvss_n', 'av_n']]

clf = XGBClassifier(n_estimators=100, max_depth=4, 
                    eval_metric='logloss', random_state=42, verbose=0)

oof_stage1 = np.zeros(len(df))
fold_aucs = []

print("Cross-Validation (5-fold, grouped by graph):")
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), 1):
    clf.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_stage1[test_idx] = clf.predict_proba(X.iloc[test_idx])[:, 1]
    fold_auc = roc_auc_score(y.iloc[test_idx], oof_stage1[test_idx])
    fold_aucs.append(fold_auc)
    print(f"  Fold {fold_num}: AUC = {fold_auc:.4f}")

# Final training for weights
clf.fit(X, y)
weights = clf.feature_importances_
weights_norm = weights / weights.sum()

print(f"\n✓ EPSS+CVSS+AV Results:")
print(f"  Mean AUC: {np.mean(fold_aucs):.4f}")
print(f"  Std AUC:  {np.std(fold_aucs):.4f}")

print(f"\n✓ Learned Feature Weights:")
print(f"  EPSS: {weights_norm[0]:.4f} ({weights_norm[0]*100:.1f}%)")
print(f"  CVSS: {weights_norm[1]:.4f} ({weights_norm[1]*100:.1f}%)")
print(f"  AV:   {weights_norm[2]:.4f} ({weights_norm[2]*100:.1f}%)")

df['score_stage1'] = oof_stage1

# ═════════════════════════════════════════════════
# COMPARISON
# ═════════════════════════════════════════════════

print("\n\n3. COMPARISON: EPSS ONLY vs EPSS+CVSS+AV")


improvement = ((np.mean(fold_aucs) - np.mean(fold_aucs_epss)) / np.mean(fold_aucs_epss) * 100)

print(f"\n{'Metric':<25} {'EPSS Only':>15} {'EPSS+CVSS+AV':>15} {'Improvement':>15}")

print(f"{'Mean AUC':<25} {np.mean(fold_aucs_epss):>15.4f} {np.mean(fold_aucs):>15.4f} {improvement:>14.2f}%")
print(f"{'Std AUC':<25} {np.std(fold_aucs_epss):>15.4f} {np.std(fold_aucs):>15.4f}")
print(f"{'Min AUC':<25} {np.min(fold_aucs_epss):>15.4f} {np.min(fold_aucs):>15.4f}")
print(f"{'Max AUC':<25} {np.max(fold_aucs_epss):>15.4f} {np.max(fold_aucs):>15.4f}")

print(f"\n✓ Summary:")
print(f"  Adding CVSS and AV improves AUC by {improvement:.2f}% over EPSS alone")
print(f"  CVSS contributes {weights_norm[1]*100:.1f}% importance")
print(f"  AV contributes {weights_norm[2]*100:.1f}% importance")

STAGE 1: TRAINING XGBOOST (WITH EPSS BASELINE)

1. BASELINE: EPSS ONLY
----------------------------------------------------------------------
Cross-Validation (5-fold, grouped by graph):
  Fold 1: AUC = 0.8377
  Fold 2: AUC = 0.8332
  Fold 3: AUC = 0.8719
  Fold 4: AUC = 0.8613
  Fold 5: AUC = 0.8345

✓ EPSS Only Results:
  Mean AUC: 0.8477
  Std AUC:  0.0158


2. FULL MODEL: EPSS + CVSS + AV
----------------------------------------------------------------------
Cross-Validation (5-fold, grouped by graph):
  Fold 1: AUC = 0.8885
  Fold 2: AUC = 0.8965
  Fold 3: AUC = 0.9016
  Fold 4: AUC = 0.9111
  Fold 5: AUC = 0.9003

✓ EPSS+CVSS+AV Results:
  Mean AUC: 0.8996
  Std AUC:  0.0073

✓ Learned Feature Weights:
  EPSS: 0.7670 (76.7%)
  CVSS: 0.1462 (14.6%)
  AV:   0.0868 (8.7%)


3. COMPARISON: EPSS ONLY vs EPSS+CVSS+AV

Metric                          EPSS Only    EPSS+CVSS+AV     Improvement
Mean AUC                           0.8477          0.8996           6.12%
Std AUC               

In [37]:

print("STAGE 2: COMPUTING NETWORK-AWARE SCORES")

#HEREEEEEEEEEEEEEEEEEEEE

# Formula: Stage2 = Stage1 × (1 + asset_criticality)

# df['score_stage2_raw'] = df['score_stage1'] * (df['asset_criticality'])

df['score_stage2_raw'] = df['score_stage1'] * (1.5 + df['asset_criticality'])

# Normalize per graph to [0, 1]
normalized_dfs = []
for graph_id, group in df.groupby('graph_id'):
    group = group.copy()
    min_score = group['score_stage2_raw'].min()
    max_score = group['score_stage2_raw'].max()
    
    if max_score > min_score:
        group['score_stage2'] = (group['score_stage2_raw'] - min_score) / (max_score - min_score)
    else:
        group['score_stage2'] = 0.5
    
    normalized_dfs.append(group)

df = pd.concat(normalized_dfs).reset_index(drop=True)

print(f"✓ Formula applied: Score_Stage2 = Score_Stage1 × (1 + asset_criticality)")
print(f"  Then normalized per graph to [0,1]")

# Compare by zone
print(f"\nScore comparison by zone:")
print(f"{'Zone':<12} {'Stage1_mean':<15} {'Stage2_mean':<15} {'Change':<15}")
print("-" * 55)
for zone in sorted(df['zone'].unique()):
    zone_df = df[df['zone'] == zone]
    s1 = zone_df['score_stage1'].mean()
    s2 = zone_df['score_stage2'].mean()
    print(f"{zone:<12} {s1:<15.4f} {s2:<15.4f} {s2-s1:+.4f}")

STAGE 2: COMPUTING NETWORK-AWARE SCORES
✓ Formula applied: Score_Stage2 = Score_Stage1 × (1 + asset_criticality)
  Then normalized per graph to [0,1]

Score comparison by zone:
Zone         Stage1_mean     Stage2_mean     Change         
-------------------------------------------------------
critical     0.0952          0.0966          +0.0014
dmz          0.0959          0.0913          -0.0047
edge         0.1046          0.0912          -0.0133
internal     0.0982          0.0959          -0.0023


In [38]:

print("STAGE 3: COMPUTING RANKS AND RANK CHANGES")


# Rank within each graph
rank_dfs = []
for graph_id, group in df.groupby('graph_id'):
    group = group.copy()
    group['rank_stage1'] = group['score_stage1'].rank(method='min', ascending=False).astype(int)
    group['rank_stage2'] = group['score_stage2'].rank(method='min', ascending=False).astype(int)
    group['rank_change'] = group['rank_stage1'] - group['rank_stage2']
    rank_dfs.append(group)

df = pd.concat(rank_dfs).reset_index(drop=True)

# Statistics
pct_changed = (df['rank_change'] != 0).sum() / len(df) * 100
moved_up = (df['rank_change'] > 0).sum()
moved_down = (df['rank_change'] < 0).sum()
avg_change = df[df['rank_change'] != 0]['rank_change'].abs().mean()

print(f"✓ Overall Reranking Statistics:")
print(f"  Total CVEs:          {len(df)}")
print(f"  CVEs that changed:   {(df['rank_change'] != 0).sum()} ({pct_changed:.2f}%)")
print(f"  Moved UP:            {moved_up}")
print(f"  Moved DOWN:          {moved_down}")
print(f"  Avg position change: {avg_change:.2f}")

print(f"\nRank change by ZONE:")
print(f"{'Zone':<12} {'Moved_UP':<10} {'Moved_DOWN':<10} {'Avg_Change':<12}")
print("-" * 50)
for zone in sorted(df['zone'].unique()):
    zone_df = df[df['zone'] == zone]
    up = (zone_df['rank_change'] > 0).sum()
    down = (zone_df['rank_change'] < 0).sum()
    avg = zone_df[zone_df['rank_change'] != 0]['rank_change'].abs().mean()
    print(f"{zone:<12} {up:<10} {down:<10} {avg:<12.2f}")

print(f"\nRank change by TOPOLOGY:")
for topo in sorted(df['topology'].unique()):
    topo_df = df[df['topology'] == topo]
    changed = (topo_df['rank_change'] != 0).sum()
    pct = changed / len(topo_df) * 100
    print(f"  {topo}: {pct:.1f}% changed rank")

STAGE 3: COMPUTING RANKS AND RANK CHANGES
✓ Overall Reranking Statistics:
  Total CVEs:          9868
  CVEs that changed:   4008 (40.62%)
  Moved UP:            2189
  Moved DOWN:          1819
  Avg position change: 1.74

Rank change by ZONE:
Zone         Moved_UP   Moved_DOWN Avg_Change  
--------------------------------------------------
critical     1981       0          1.64        
dmz          0          268        1.55        
edge         0          855        2.69        
internal     208        696        1.12        

Rank change by TOPOLOGY:
  cloud: 53.4% changed rank
  enterprise: 30.0% changed rank


In [39]:
print("="*70)
print("COMPREHENSIVE EVALUATION: EPSS ONLY vs STAGE 1 vs STAGE 2")
print("="*70)

def evaluate_ranking(df_eval, score_col):
    """Evaluate ranking quality within each graph"""
    aucs, p10, p20, r10, r20 = [], [], [], [], []
    
    for graph_id, group in df_eval.groupby('graph_id'):
        if group['in_kev'].nunique() < 2:
            continue
        
        aucs.append(roc_auc_score(group['in_kev'], group[score_col]))
        
        total = group['in_kev'].sum()
        top10 = group.nlargest(10, score_col)
        top20 = group.nlargest(20, score_col)
        
        p10.append(top10['in_kev'].sum() / 10)
        p20.append(top20['in_kev'].sum() / 20)
        r10.append(top10['in_kev'].sum() / total if total else 0)
        r20.append(top20['in_kev'].sum() / total if total else 0)
    
    return {
        'AUC': np.mean(aucs),
        'AUC_std': np.std(aucs),
        'P@10': np.mean(p10),
        'P@20': np.mean(p20),
        'R@10': np.mean(r10),
        'R@20': np.mean(r20),
    }

# Evaluate all three
results_epss = evaluate_ranking(df, 'score_epss_only')
results_s1 = evaluate_ranking(df, 'score_stage1')
results_s2 = evaluate_ranking(df, 'score_stage2')

# Main comparison table
print("\n" + "="*90)
print(f"{'Metric':<10} {'EPSS Only':>15} {'Stage 1':>15} {'Stage 2':>15}")
print("="*90)

metrics = ['AUC', 'P@10', 'P@20', 'R@10', 'R@20']
for metric in metrics:
    v_epss = results_epss[metric]
    v_s1 = results_s1[metric]
    v_s2 = results_s2[metric]
    print(f"{metric:<10} {v_epss:>15.4f} {v_s1:>15.4f} {v_s2:>15.4f}")

print("="*90)

# Improvement analysis
print("\n" + "="*90)
print("IMPROVEMENT OVER EPSS ONLY")
print("="*90)
print(f"\n{'Metric':<10} {'Stage 1 vs EPSS':>20} {'Stage 2 vs EPSS':>20}")
print("-" * 55)

for metric in metrics:
    v_epss = results_epss[metric]
    v_s1 = results_s1[metric]
    v_s2 = results_s2[metric]
    
    improve_s1 = ((v_s1 - v_epss) / v_epss * 100) if v_epss != 0 else 0
    improve_s2 = ((v_s2 - v_epss) / v_epss * 100) if v_epss != 0 else 0
    
    print(f"{metric:<10} {improve_s1:>19.2f}% {improve_s2:>19.2f}%")

print("="*90)

# Stage 2 improvement over Stage 1
print("\n" + "="*90)
print("STAGE 2 vs STAGE 1 (Network Structure Impact)")
print("="*90)
print(f"\n{'Metric':<10} {'Stage 1':>15} {'Stage 2':>15} {'Improvement':>15}")
print("-" * 55)

for metric in metrics:
    v_s1 = results_s1[metric]
    v_s2 = results_s2[metric]
    improve = ((v_s2 - v_s1) / v_s1 * 100) if v_s1 != 0 else 0
    print(f"{metric:<10} {v_s1:>15.4f} {v_s2:>15.4f} {improve:>14.2f}%")

print("="*90)

# By topology
print("\n" + "="*90)
print("EVALUATION BY TOPOLOGY")
print("="*90)

for topo in sorted(df['topology'].unique()):
    topo_df = df[df['topology'] == topo]
    res_epss = evaluate_ranking(topo_df, 'score_epss_only')
    res_s1 = evaluate_ranking(topo_df, 'score_stage1')
    res_s2 = evaluate_ranking(topo_df, 'score_stage2')
    
    print(f"\n{topo.upper()}:")
    print(f"  EPSS Only:     AUC = {res_epss['AUC']:.4f}")
    print(f"  Stage 1:       AUC = {res_s1['AUC']:.4f}  (+{((res_s1['AUC']-res_epss['AUC'])/res_epss['AUC']*100):.2f}%)")
    print(f"  Stage 2:       AUC = {res_s2['AUC']:.4f}  (+{((res_s2['AUC']-res_epss['AUC'])/res_epss['AUC']*100):.2f}%)")

print("\n" + "="*90)

COMPREHENSIVE EVALUATION: EPSS ONLY vs STAGE 1 vs STAGE 2

Metric           EPSS Only         Stage 1         Stage 2
AUC                 0.8544          0.9044          0.9039
P@10                0.7186          0.7171          0.7143
P@20                0.4671          0.4779          0.4771
R@10                0.5430          0.5464          0.5419
R@20                0.6913          0.7126          0.7120

IMPROVEMENT OVER EPSS ONLY

Metric          Stage 1 vs EPSS      Stage 2 vs EPSS
-------------------------------------------------------
AUC                       5.85%                5.80%
P@10                     -0.20%               -0.60%
P@20                      2.29%                2.14%
R@10                      0.64%               -0.20%
R@20                      3.09%                2.99%

STAGE 2 vs STAGE 1 (Network Structure Impact)

Metric             Stage 1         Stage 2     Improvement
-------------------------------------------------------
AUC                 0

In [40]:
print("="*70)
print("CASE STUDY: INTERESTING RANK MOVEMENTS")
print("="*70)

# Find CVEs that moved significantly
significant = df[df['rank_change'].abs() >= 5].copy()
print(f"\nCVEs with ≥5 position change: {len(significant)}")

# Find a KEV that moved
kev_movers = significant[significant['in_kev'] == 1]
if len(kev_movers) > 0:
    example = kev_movers.iloc[0]
else:
    example = significant.iloc[0]

print(f"\nExample CVE: {example['cve']}")
print(f"  Graph ID:          {int(example['graph_id'])}")
print(f"  Topology:          {example['topology']}")
print(f"  Zone:              {example['zone']}")
print(f"  Asset Criticality: {example['asset_criticality']:.2f}")
print(f"  EPSS:              {example['epss_n']:.4f}")
print(f"  CVSS:              {example['cvss_n']:.4f}")
print(f"  AV:                {example['av_n']:.4f}")
print(f"  Stage 1 Score:     {example['score_stage1']:.4f}")
print(f"  Stage 2 Score:     {example['score_stage2']:.4f}")
print(f"  Rank Stage 1:      {int(example['rank_stage1'])}")
print(f"  Rank Stage 2:      {int(example['rank_stage2'])}")
print(f"  Rank Change:       {int(example['rank_change'])} positions (moved UP)" if example['rank_change'] > 0 else f"  Rank Change:       {int(example['rank_change'])} positions (moved DOWN)")
print(f"  Is KEV?            {'YES' if example['in_kev'] == 1 else 'NO'}")

# Show top movers
print(f"\n\nTop 5 UPWARD movers:")
top_up = df[df['rank_change'] > 0].nlargest(5, 'rank_change')[['cve', 'zone', 'rank_stage1', 'rank_stage2', 'rank_change', 'in_kev']]
print(top_up.to_string())

print(f"\nTop 5 DOWNWARD movers:")
top_down = df[df['rank_change'] < 0].nsmallest(5, 'rank_change')[['cve', 'zone', 'rank_stage1', 'rank_stage2', 'rank_change', 'in_kev']]
print(top_down.to_string())

CASE STUDY: INTERESTING RANK MOVEMENTS

CVEs with ≥5 position change: 154

Example CVE: CVE-2010-0249
  Graph ID:          8
  Topology:          cloud
  Zone:              edge
  Asset Criticality: 0.10
  EPSS:              1.0000
  CVSS:              0.8405
  AV:                0.2385
  Stage 1 Score:     0.9995
  Stage 2 Score:     0.8652
  Rank Stage 1:      1
  Rank Stage 2:      6
  Rank Change:       -5 positions (moved DOWN)
  Is KEV?            YES


Top 5 UPWARD movers:
                 cve      zone  rank_stage1  rank_stage2  rank_change  in_kev
4728   CVE-2013-2138  critical           48           40            8       0
4729   CVE-2019-1130  critical           60           53            7       1
4730  CVE-2021-20190  critical           67           60            7       0
5909  CVE-2020-19213  critical           75           68            7       0
2221   CVE-2024-8007  critical           88           82            6       0

Top 5 DOWNWARD movers:
                 cve  z

In [41]:
upward = df[df['rank_change'] > 0]
downward = df[df['rank_change'] < 0]
unchanged = df[df['rank_change'] == 0]

print("="*70)
print("KEV DETECTION BY MOVEMENT CATEGORY")
print("="*70)

print(f"\nMoved UP (n={len(upward)}):")
print(f"  KEVs: {upward['in_kev'].sum()}/{len(upward)} = {upward['in_kev'].mean()*100:.1f}%")

print(f"\nMoved DOWN (n={len(downward)}):")
print(f"  KEVs: {downward['in_kev'].sum()}/{len(downward)} = {downward['in_kev'].mean()*100:.1f}%")

print(f"\nUnchanged (n={len(unchanged)}):")
print(f"  KEVs: {unchanged['in_kev'].sum()}/{len(unchanged)} = {unchanged['in_kev'].mean()*100:.1f}%")

KEV DETECTION BY MOVEMENT CATEGORY

Moved UP (n=2189):
  KEVs: 219/2189 = 10.0%

Moved DOWN (n=1819):
  KEVs: 194/1819 = 10.7%

Unchanged (n=5860):
  KEVs: 567/5860 = 9.7%


In [42]:
import numpy as np
import pandas as pd

ks = [5, 10, 15, 20]

kev_before = {k: [] for k in ks}
kev_after  = {k: [] for k in ks}

for _, group in df.groupby('graph_id'):
    if group['in_kev'].nunique() < 2:
        continue
    for k in ks:
        kev_before[k].append(
            group.nlargest(k, 'score_stage1')['in_kev'].sum()
        )
        kev_after[k].append(
            group.nlargest(k, 'score_stage2')['in_kev'].sum()
        )

print("=" * 55)
print(f"{'K':<6} {'KEV Before':>12} {'KEV After':>12} {'Improvement':>12}")
print("=" * 55)
for k in ks:
    before = np.mean(kev_before[k])
    after  = np.mean(kev_after[k])
    improvement = ((after - before) / before * 100) if before > 0 else 0
    print(f"@{k:<5} {before:>12.2f} {after:>12.2f} {improvement:>11.1f}%")
print("=" * 55)

K        KEV Before    KEV After  Improvement
@5             4.56         4.50        -1.3%
@10            7.17         7.14        -0.4%
@15            8.59         8.60         0.2%
@20            9.56         9.54        -0.1%


In [43]:
import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════
# PICK ONE GRAPH TO INSPECT
# ═══════════════════════════════════════════════
GRAPH_ID = 67  # change this to inspect different graphs

group = df[df['graph_id'] == GRAPH_ID].copy()

print(f"Graph {GRAPH_ID} — Topology: {group['topology'].iloc[0]}")
print(f"Total CVEs: {len(group)}")
print(f"KEV: {group['in_kev'].sum()}, Non-KEV: {(group['in_kev']==0).sum()}")
print(f"Zones: {group['zone'].value_counts().to_dict()}")

# ═══════════════════════════════════════════════
# COMPUTE RANKINGS
# ═══════════════════════════════════════════════
group['rank_before'] = group['score_stage1'].rank(ascending=False).astype(int)
group['rank_after']  = group['score_stage2'].rank(ascending=False).astype(int)
group['rank_change'] = group['rank_before'] - group['rank_after']

# ═══════════════════════════════════════════════
# FULL RANKED LIST — BEFORE
# ═══════════════════════════════════════════════
print("\n" + "=" * 90)
print("RANKING BEFORE (Stage 1 — EPSS+CVSS+AV only)")
print("=" * 90)
print(f"{'Rank':<6} {'CVE':<20} {'Zone':<12} {'AC':<6} {'EPSS':>8} {'CVSS':>8} {'AV':>6} {'Score':>8} {'KEV':>6}")
print("-" * 90)
before_sorted = group.sort_values('rank_before')
for _, row in before_sorted.iterrows():
    kev_flag = '✅' if row['in_kev'] == 1 else '❌'
    print(f"{int(row['rank_before']):<6} {row['cve']:<20} {row['zone']:<12} "
          f"{row['asset_criticality']:<6.1f} {row['epss_n']:>8.3f} "
          f"{row['cvss_n']:>8.3f} {row['av_n']:>6.2f} "
          f"{row['score_stage1']:>8.3f} {kev_flag:>6}")

# ═══════════════════════════════════════════════
# FULL RANKED LIST — AFTER
# ═══════════════════════════════════════════════
print("\n" + "=" * 90)
print("RANKING AFTER (Stage 2 — Stage1 × (1 + AC))")
print("=" * 90)
print(f"{'Rank':<6} {'CVE':<20} {'Zone':<12} {'AC':<6} {'EPSS':>8} {'CVSS':>8} {'AV':>6} {'Score':>8} {'KEV':>6} {'Change':>8}")
print("-" * 90)
after_sorted = group.sort_values('rank_after')
for _, row in after_sorted.iterrows():
    kev_flag  = '✅' if row['in_kev'] == 1 else '❌'
    change    = int(row['rank_change'])
    arrow     = f"↑{change}" if change > 0 else (f"↓{abs(change)}" if change < 0 else "—")
    print(f"{int(row['rank_after']):<6} {row['cve']:<20} {row['zone']:<12} "
          f"{row['asset_criticality']:<6.1f} {row['epss_n']:>8.3f} "
          f"{row['cvss_n']:>8.3f} {row['av_n']:>6.2f} "
          f"{row['score_stage2']:>8.3f} {kev_flag:>6} {arrow:>8}")

# ═══════════════════════════════════════════════
# BIGGEST MOVERS
# ═══════════════════════════════════════════════
print("\n" + "=" * 70)
print("TOP 5 UPWARD MOVERS (gained priority due to network position)")
print("=" * 70)
print(f"{'CVE':<20} {'Zone':<12} {'AC':<6} {'Before':<10} {'After':<10} {'Change'}")
print("-" * 70)
top_up = group.nlargest(5, 'rank_change')
for _, row in top_up.iterrows():
    kev = '✅' if row['in_kev'] == 1 else '❌'
    print(f"{row['cve']:<20} {row['zone']:<12} {row['asset_criticality']:<6.1f} "
          f"{int(row['rank_before']):<10} {int(row['rank_after']):<10} "
          f"+{int(row['rank_change'])} {kev}")

print("\n" + "=" * 70)
print("TOP 5 DOWNWARD MOVERS (lost priority due to network position)")
print("=" * 70)
print(f"{'CVE':<20} {'Zone':<12} {'AC':<6} {'Before':<10} {'After':<10} {'Change'}")
print("-" * 70)
top_down = group.nsmallest(5, 'rank_change')
for _, row in top_down.iterrows():
    kev = '✅' if row['in_kev'] == 1 else '❌'
    print(f"{row['cve']:<20} {row['zone']:<12} {row['asset_criticality']:<6.1f} "
          f"{int(row['rank_before']):<10} {int(row['rank_after']):<10} "
          f"{int(row['rank_change'])} {kev}")

# ═══════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════
print("\n" + "=" * 50)
print(f"SUMMARY — Graph {GRAPH_ID}")
print("=" * 50)
total     = len(group)
reranked  = (group['rank_change'] != 0).sum()
print(f"Total CVEs:        {total}")
print(f"Reranked:          {reranked} ({reranked/total*100:.1f}%)")
print(f"Moved up:          {(group['rank_change'] > 0).sum()}")
print(f"Moved down:        {(group['rank_change'] < 0).sum()}")
print(f"Unchanged:         {(group['rank_change'] == 0).sum()}")
print(f"\nAvg rank change by zone:")
print(group.groupby('zone')['rank_change'].mean().sort_values(ascending=False).round(2))
print(f"\nKEV in top 10 before: {group.nlargest(10,'score_stage1')['in_kev'].sum()}")
print(f"KEV in top 10 after:  {group.nlargest(10,'score_stage2')['in_kev'].sum()}")

Graph 67 — Topology: cloud
Total CVEs: 165
KEV: 7, Non-KEV: 158
Zones: {'critical': 135, 'edge': 30}

RANKING BEFORE (Stage 1 — EPSS+CVSS+AV only)
Rank   CVE                  Zone         AC         EPSS     CVSS     AV    Score    KEV
------------------------------------------------------------------------------------------
1      CVE-2023-44221       critical     0.3       0.826    0.651   0.44    0.980      ✅
2      CVE-2021-38647       critical     0.3       1.000    1.000   0.51    0.846      ✅
3      CVE-2015-6967        critical     0.3       0.386    0.497   0.38    0.653      ❌
4      CVE-2020-8264        critical     0.3       0.728    0.404   0.33    0.491      ❌
5      CVE-2004-0487        critical     0.3       0.070    1.000   0.31    0.467      ❌
6      CVE-2019-11001       critical     0.3       0.470    0.480   0.36    0.312      ✅
7      CVE-2023-29340       edge         0.1       0.084    0.921   0.16    0.288      ❌
8      CVE-2024-21410       critical     0.3      